[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShintaroMinami/notebook_proteina_complexa/blob/main/proteina_complexa.ipynb)

# Proteina Complexa — Hands-On Notebook

このノートブックでは、NVIDIAが開発したタンパク質デザインモデル **Proteina Complexa** を
Google Colab上で実行します。

> **必要なもの:** Googleアカウントのみ。ローカル環境の構築は不要です。

In [ ]:
#@title 初期設定
#@markdown <small>このセルを最初に実行してください。</small>
#@markdown ---
#@markdown ### Google Drive の使用
#@markdown <small>モデルの重み（約 13〜25 GB）を Google Drive に保存し、次回以降のダウンロードを省略します。</small>
#@markdown - **ON**: 初回はダウンロード後 Drive に保存。2回目以降は Drive から復元（高速）
#@markdown - **OFF**: 毎回インターネットからダウンロード
use_google_drive = False #@param {type:"boolean"}

import subprocess
from IPython.display import display, HTML

def run(cmd, title=""):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    lines = output.split("\n")
    icon = "\u2705" if result.returncode == 0 else "\u274c"
    summary = f"{icon} {title} ({len(lines)} lines)" if title else f"{icon} Output ({len(lines)} lines)"
    html = (
        f'<details><summary style="cursor:pointer;font-weight:bold;padding:4px 0">{summary}</summary>'
        f'<pre style="max-height:300px;overflow-y:auto;background:#f5f5f5;padding:8px;'
        f'font-size:11px;border-radius:4px;margin-top:4px">{output}</pre>'
        f'</details>'
    )
    display(HTML(html))
    return result.returncode

if use_google_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_CACHE = "/content/drive/MyDrive/.proteina_complexa_cache"
    import os
    os.makedirs(DRIVE_CACHE, exist_ok=True)
    print(f"Google Drive cache: {DRIVE_CACHE}")
else:
    DRIVE_CACHE = None
    print("Google Drive: OFF（毎回ダウンロードします）")

print("Ready.")


---
<h1 style="border-bottom: 3px solid #4285f4; padding-bottom: 8px; color: #1a73e8;">1. 環境構築</h1>

はじめに、必要なソフトウェアをインストールします。
以下のセルを **上から順に** 実行してください（各セルで Shift+Enter）。

In [ ]:
#@title 1.1 GPU の確認
#@markdown <small>まず、GPUが割り当てられているか確認します。</small>
#@markdown <small>「GPU が見つかりません」と表示された場合は、メニューの **「ランタイム」→「ランタイムのタイプを変更」** から **GPU** を選択してください。</small>
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode == 0:
    gpu_info = result.stdout.strip()
    print(f"GPU: {gpu_info}")
else:
    print("GPU が見つかりません。ランタイムのタイプを変更してください。")

print(f"Python: {sys.version}")

In [ ]:
#@title 1.2 ソフトウェアのインストール
#@markdown <small>uv（高速パッケージマネージャ）、Proteina Complexa、PyTorch、Boltz-2 をインストールします。</small>
#@markdown <small>このセルの実行には **5〜10分** ほどかかります。</small>
import os

REPO_DIR = "/content/Proteina-Complexa"

# --- uv + repository ---
run("pip install -q uv", "uv install")

if os.path.exists(REPO_DIR):
    print(f"Repository already exists: {REPO_DIR}")
else:
    run("git clone --depth 1 https://github.com/NVIDIA-Digital-Bio/Proteina-Complexa.git /content/Proteina-Complexa", "git clone")

os.chdir(REPO_DIR)
print(f"リポジトリ: {os.getcwd()}")

# --- PyTorch ---
run(
    "uv pip install --system torch==2.7.0+cu126 torchvision torchaudio"
    " --index-url https://download.pytorch.org/whl/cu126",
    "PyTorch + CUDA 12.6"
)

# --- Proteina Complexa ---
run("uv pip install --system --index-strategy unsafe-best-match -e .", "Proteina Complexa")
run(
    "uv pip install --system torch_geometric torch_scatter torch_sparse torch_cluster"
    " -f https://data.pyg.org/whl/torch-2.7.0+cu126.html",
    "PyTorch Geometric"
)
run("uv pip install --system graphein==1.7.7 --no-deps", "Graphein")
run("uv pip install --system \"atomworks[ml,openbabel,dev]\" 2>/dev/null || true", "Atomworks")
run("uv pip install --system py3Dmol", "py3Dmol")
run("uv pip install --system dm-haiku jax", "dm-haiku + JAX")
run("uv pip install --system biotite==1.6.0", "biotite")

# --- Boltz-2 ---
run("uv pip install --system --index-strategy unsafe-best-match \"boltz[cuda]\"", "Boltz-2")
run(
    "uv pip install --system --reinstall torch==2.7.0+cu126 torchvision torchaudio"
    " --index-url https://download.pytorch.org/whl/cu126",
    "PyTorch reinstall"
)
run("uv pip install --system \"numpy<2.1\"", "NumPy pin")

import torch
print(f"\nPyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print("Done.")


In [ ]:
#@title 1.3 モデルの重みをダウンロード
#@markdown <small>ダウンロードするモデルを選択してください。Google Drive を有効にしている場合、2回目以降は Drive から復元します。</small>
#@markdown ---
#@markdown | モデル | 用途 | サイズ |
#@markdown |---|---|---|
#@markdown | Protein Target | タンパク質に結合するバインダーの設計 | 約 6.6 GB |
#@markdown | Ligand Target | 低分子リガンドに結合するバインダーの設計 | 約 5.9 GB |
#@markdown | AME | 機能モチーフのスキャフォールディング | 約 5.9 GB |
#@markdown | Boltz-2 | 構造予測・検証 | 約 6.2 GB |
#@markdown ---
download_protein_target = True #@param {type:"boolean"}
download_ligand_target = False #@param {type:"boolean"}
download_ame = False #@param {type:"boolean"}
download_boltz2 = True #@param {type:"boolean"}

import os, shutil
from pathlib import Path

# --- Proteina Complexa ---
if not os.path.exists(".env"):
    run("complexa init", "Proteina Complexa init")
else:
    print("Proteina Complexa: .env already exists, skipping init.")

os.environ["COMPLEXA_INIT"] = "1"
os.environ["LOCAL_CODE_PATH"] = "/content/Proteina-Complexa"
os.environ["DATA_PATH"] = "/content/Proteina-Complexa/assets"

complexa_models = []
if download_protein_target:
    complexa_models.append("--complexa")
if download_ligand_target:
    complexa_models.append("--complexa-ligand")
if download_ame:
    complexa_models.append("--complexa-ame")

if complexa_models:
    dl_flags = " ".join(complexa_models)
    complexa_cache = Path(DRIVE_CACHE) / "complexa" if DRIVE_CACHE else None
    complexa_local = Path("checkpoints")

    if complexa_cache and complexa_cache.exists() and any(complexa_cache.iterdir()):
        print("Proteina Complexa weights: restoring from Google Drive...")
        if complexa_local.exists():
            shutil.rmtree(complexa_local)
        shutil.copytree(complexa_cache, complexa_local)
        print("Proteina Complexa weights: restored.")
    else:
        run(f"complexa download {dl_flags}", "Proteina Complexa weights")
        if complexa_cache and complexa_local.exists():
            print("Saving Proteina Complexa weights to Google Drive...")
            if complexa_cache.exists():
                shutil.rmtree(complexa_cache)
            shutil.copytree(complexa_local, complexa_cache)
            print("Saved.")

# --- Boltz-2 ---
if download_boltz2:
    boltz_cache = Path(DRIVE_CACHE) / "boltz" if DRIVE_CACHE else None
    boltz_local = Path("/root/.boltz")

    if boltz_cache and boltz_cache.exists() and any(boltz_cache.iterdir()):
        print("Boltz-2 weights: linking from Google Drive...")
        if boltz_local.exists() or boltz_local.is_symlink():
            boltz_local.unlink() if boltz_local.is_symlink() else shutil.rmtree(boltz_local)
        boltz_local.symlink_to(boltz_cache)
        print("Boltz-2 weights: ready (symlink).")
    else:
        run("python -c \"from pathlib import Path; p = Path('/root/.boltz'); p.mkdir(parents=True, exist_ok=True); from boltz.main import download_boltz2; download_boltz2(p)\"", "Boltz-2 download")
        if boltz_cache and boltz_local.exists():
            print("Saving Boltz-2 weights to Google Drive...")
            if boltz_cache.exists():
                shutil.rmtree(boltz_cache)
            shutil.copytree(boltz_local, boltz_cache)
            shutil.rmtree(boltz_local)
            boltz_local.symlink_to(boltz_cache)
            print("Saved.")

print("All selected models ready.")

In [ ]:
#@title 1.4 インストールの確認
#@markdown <small>ソフトウェアとモデルの重みが正しくインストールされたか確認します。</small>
from pathlib import Path
import torch

all_ok = True

# --- Software ---
print("Software")
print("─" * 40)
print(f"  PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

import shutil
if shutil.which("complexa"):
    print("  Proteina Complexa CLI: OK")
else:
    print("  Proteina Complexa CLI: NOT FOUND")
    all_ok = False

# --- Model weights ---
print()
print("Model weights")
print("─" * 40)

ckpt_dir = Path("/content/Proteina-Complexa/checkpoints")
weight_checks = {
    "Protein Target": ("complexa.ckpt", download_protein_target),
    "Ligand Target": ("complexa_ligand.ckpt", download_ligand_target),
    "AME": ("complexa_ame.ckpt", download_ame),
}

for name, (filename, requested) in weight_checks.items():
    if not requested:
        continue
    path = ckpt_dir / filename
    if path.exists():
        size_gb = path.stat().st_size / (1024**3)
        print(f"  {name}: OK ({size_gb:.1f} GB)")
    else:
        print(f"  {name}: NOT FOUND ({path})")
        all_ok = False

if download_boltz2:
    boltz_dir = Path("/root/.boltz")
    if boltz_dir.exists() and any(boltz_dir.rglob("*.pt")):
        total = sum(f.stat().st_size for f in boltz_dir.rglob("*") if f.is_file())
        print(f"  Boltz-2: OK ({total / (1024**3):.1f} GB)")
    else:
        print(f"  Boltz-2: NOT FOUND ({boltz_dir})")
        all_ok = False

print()
if all_ok:
    print("All checks passed!")
else:
    print("Some checks failed. Please re-run the corresponding cells above.")


---
<h1 style="border-bottom: 3px solid #4285f4; padding-bottom: 8px; color: #1a73e8;">2. プロジェクトの準備</h1>


In [ ]:
#@title 2.0 プロジェクトの作成
#@markdown <small>デザインの結果を保存するプロジェクトを作成します。プロジェクト名を入力してください（英数字・アンダースコア推奨）。</small>
#@markdown ---
project_name = "PDL1_binder_design" #@param {type:"string"}

from pathlib import Path

PROJECT_DIR = Path(f"/content/projects/{project_name}")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "targets").mkdir(exist_ok=True)
(PROJECT_DIR / "results").mkdir(exist_ok=True)
(PROJECT_DIR / "analysis").mkdir(exist_ok=True)

print(f"Project: {project_name}")
print(f"Directory: {PROJECT_DIR}")
print(f"")
print(f"  targets/   ... ターゲットPDBファイル")
print(f"  results/   ... デザイン結果")
print(f"  analysis/  ... 解析結果")

In [ ]:
#@title 2.1 ターゲットの指定
#@markdown <small>ターゲットタンパク質の取得方法を選択してください。</small>
#@markdown - **PDB ID**: RCSB PDBからダウンロードします（例: `3BIK`, `7BQD`）
#@markdown - **Upload**: 手元のPDBファイルをアップロードします
#@markdown ---
target_source = "PDB ID" #@param ["PDB ID", "Upload"]
pdb_id = "3BIK" #@param {type:"string"}

from pathlib import Path

target_dir = PROJECT_DIR / "targets"

if target_source == "PDB ID":
    pdb_id = pdb_id.strip().upper()
    target_pdb = target_dir / f"{pdb_id}.pdb"
    if target_pdb.exists():
        print(f"Already downloaded: {target_pdb}")
    else:
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        run(f"wget -q -O {target_pdb} {url}", f"Download {pdb_id}")
        if target_pdb.exists() and target_pdb.stat().st_size > 0:
            print(f"Downloaded: {target_pdb}")
        else:
            print(f"Error: PDB ID '{pdb_id}' が見つかりませんでした。IDを確認してください。")
else:
    from google.colab import files
    print("PDBファイルを選択してください:")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        target_pdb = target_dir / filename
        target_pdb.write_bytes(content)
        pdb_id = target_pdb.stem.upper()
        print(f"Uploaded: {target_pdb}")

print(f"\nTarget PDB: {target_pdb}")

In [ ]:
#@title 2.2 ターゲット構造の確認
#@markdown <small>ターゲットPDBの3D構造とチェーン情報を表示します。</small>
from Bio.PDB import PDBParser
import py3Dmol
import warnings
warnings.filterwarnings("ignore")

# Chain info
parser = PDBParser(QUIET=True)
structure = parser.get_structure("target", str(target_pdb))

print(f"File: {target_pdb.name}")
print("─" * 50)
for model in structure:
    for chain in model:
        residues = [r for r in chain.get_residues() if r.id[0] == " "]
        if not residues:
            continue
        first = residues[0].id[1]
        last = residues[-1].id[1]
        print(f"  Chain {chain.id}: residues {first}-{last} ({len(residues)} residues)")
    break
print("─" * 50)

# 3D viewer
with open(target_pdb) as f:
    pdb_data = f.read()

view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_data, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()

In [ ]:
#@title 2.3 バインダー設計パラメータの設定
#@markdown <small>上の構造情報を参考に、以下のパラメータを設定してください。</small>
#@markdown ---
#@markdown ### ターゲット設定
#@markdown - **target_chain_residues**: ターゲットのチェーンと残基範囲（例: `A1-115`）
#@markdown - **hotspot_residues**: バインダーが結合してほしい残基（カンマ区切り、例: `A56,A113,A121`）
#@markdown ### バインダー設定
#@markdown - **binder_length_min / max**: 生成するバインダーの長さの範囲
#@markdown ---
target_chain_residues = "A18-133" #@param {type:"string"}
hotspot_residues = "A56,A113,A115,A121,A123" #@param {type:"string"}
binder_length_min = 60 #@param {type:"integer"}
binder_length_max = 80 #@param {type:"integer"}

hotspot_list = [r.strip() for r in hotspot_residues.split(",") if r.strip()]

target_config = {
    "source": "user_targets",
    "target_filename": target_pdb.stem,
    "target_path": str(target_pdb),
    "target_input": target_chain_residues,
    "hotspot_residues": hotspot_list,
    "binder_length": [binder_length_min, binder_length_max],
    "pdb_id": pdb_id.lower(),
}

print("Target configuration:")
for k, v in target_config.items():
    print(f"  {k}: {v}")
TASK_NAME = f"user_{pdb_id}"
print(f"\nTask name: {TASK_NAME}")

# --- Parse target_chain_residues ---
import re
match = re.match(r"([A-Z])(\d+)-(\d+)", target_chain_residues)
target_chain = match.group(1)
target_resi_start = int(match.group(2))
target_resi_end = int(match.group(3))

# --- Parse hotspot residues ---
hotspot_resis = []
for h in hotspot_list:
    m = re.match(r"([A-Z])(\d+)", h)
    if m:
        hotspot_resis.append(int(m.group(2)))

# --- 3D Viewer ---
import py3Dmol

with open(target_pdb) as f:
    pdb_data = f.read()

view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_data, "pdb")

# Non-target region: thin, transparent
view.setStyle(
    {"cartoon": {"color": "lightgray", "opacity": 0.3}}
)

# Target region: colored
view.setStyle(
    {"chain": target_chain, "resi": [f"{target_resi_start}-{target_resi_end}"]},
    {"cartoon": {"color": "skyblue"}}
)

# Hotspot residues: red sticks + cartoon
if hotspot_resis:
    view.setStyle(
        {"chain": target_chain, "resi": hotspot_resis},
        {"cartoon": {"color": "red"}, "stick": {"color": "red"}}
    )

view.zoomTo({"chain": target_chain, "resi": [f"{target_resi_start}-{target_resi_end}"]})
view.show()

print(f"\nBlue: target region ({target_chain_residues})")
print(f"Red:  hotspot residues ({hotspot_residues})")
print(f"Gray: non-target region")

---
<h1 style="border-bottom: 3px solid #4285f4; padding-bottom: 8px; color: #1a73e8;">3. バインダーデザインの実行</h1>


In [ ]:
#@title 3.1 デザインパイプラインの設定
#@markdown <small>生成パラメータを設定します。</small>
#@markdown - **num_samples**: 生成するバインダー候補の数
#@markdown - **search_algorithm**: 探索アルゴリズム（デモでは single-pass 推奨）
#@markdown - **seed**: 乱数シード（再現性のため）
#@markdown - **ncpus**: 使用するCPUコア数
#@markdown ---
num_samples = 4 #@param {type:"integer"}
search_algorithm = "single-pass" #@param ["single-pass", "best-of-n"]
seed = 1101 #@param {type:"integer"}
ncpus = 2 #@param {type:"integer"}

from pathlib import Path

COMPLEXA_DIR = Path("/content/Proteina-Complexa")
RESULTS_DIR = PROJECT_DIR / "results"

# Write custom pipeline config as raw YAML (Hydra requires specific defaults format)
pipeline_yaml = f"""defaults:
  - pipeline/binder/binder_generate@generation
  - _self_

run_name: {project_name}
ckpt_path: {COMPLEXA_DIR / "ckpts"}
ckpt_name: complexa.ckpt
autoencoder_ckpt_path: {COMPLEXA_DIR / "ckpts" / "complexa_ae.ckpt"}
ncpus_: {ncpus}
seed: {seed}
gen_njobs: 1
eval_njobs: 1

hydra:
  run:
    dir: {RESULTS_DIR / "hydra_outputs"}
"""

pipeline_path = COMPLEXA_DIR / "configs" / "user_pipeline.yaml"
pipeline_path.write_text(pipeline_yaml)
print(f"Pipeline config written: {pipeline_path}")
print(f"Output directory: {RESULTS_DIR}")
print(f"Samples: {num_samples}, Algorithm: {search_algorithm}, Seed: {seed}, CPUs: {ncpus}")

In [ ]:
#@title 3.2 バインダーデザインの実行
#@markdown <small>Proteina Complexa でバインダーを生成します。T4 GPU では数分〜十数分かかります。</small>
import os

os.chdir("/content/Proteina-Complexa")

cmd = (
    f"complexa generate configs/user_pipeline.yaml"
    f" ++generation.task_name={TASK_NAME}"
    f" ++generation.search.algorithm={search_algorithm}"
    f" ++generation.dataloader.batch_size={num_samples}"
    f" ++generation.target_dict_cfg.{TASK_NAME}.source=user_targets"
    f" ++generation.target_dict_cfg.{TASK_NAME}.target_filename={target_config['target_filename']}"
    f" ++generation.target_dict_cfg.{TASK_NAME}.target_path={target_config['target_path']}"
    f" ++generation.target_dict_cfg.{TASK_NAME}.target_input={target_config['target_input']}"
    f" '++generation.target_dict_cfg.{TASK_NAME}.hotspot_residues={target_config["hotspot_residues"]}'"
    f" '++generation.target_dict_cfg.{TASK_NAME}.binder_length={target_config["binder_length"]}'"
    f" ++generation.target_dict_cfg.{TASK_NAME}.pdb_id={target_config['pdb_id']}"
    f" ++generation.reward_model=null"
)

print(f"Running: {cmd[:100]}...")
print()
run(cmd, "Binder generation")
print("Generation complete.")

In [ ]:
#@title 3.3 生成結果の確認
#@markdown <small>生成されたバインダー候補のPDBファイルを確認します。</small>
import glob
from pathlib import Path

# Find generated PDB files
output_dirs = sorted(glob.glob(f"/content/Proteina-Complexa/outputs/{project_name}/**/samples/**/*.pdb", recursive=True))
if not output_dirs:
    output_dirs = sorted(glob.glob(f"/content/Proteina-Complexa/outputs/**/*.pdb", recursive=True))

if output_dirs:
    # Copy results to project directory
    import shutil
    results_pdb_dir = PROJECT_DIR / "results" / "generated_pdbs"
    results_pdb_dir.mkdir(parents=True, exist_ok=True)
    for pdb_file in output_dirs:
        shutil.copy2(pdb_file, results_pdb_dir)
    
    print(f"Generated {len(output_dirs)} binder candidates:")
    for p in output_dirs:
        print(f"  {Path(p).name}")
    print(f"\nCopied to: {results_pdb_dir}")
else:
    print("No PDB files found. Check the generation log above for errors.")

In [ ]:
#@title 3.4 デザイン結果の3D表示
#@markdown <small>生成されたバインダーとターゲットの複合体構造を表示します。</small>
#@markdown ---
view_index = 0 #@param {type:"integer"}

import py3Dmol
from pathlib import Path

results_pdb_dir = PROJECT_DIR / "results" / "generated_pdbs"
pdb_files = sorted(results_pdb_dir.glob("*.pdb"))

if not pdb_files:
    print("PDBファイルが見つかりません。先に 3.2 を実行してください。")
elif view_index >= len(pdb_files):
    print(f"Index {view_index} is out of range. {len(pdb_files)} files available (0-{len(pdb_files)-1}).")
else:
    pdb_path = pdb_files[view_index]
    print(f"Showing: {pdb_path.name} ({view_index + 1}/{len(pdb_files)})")

    with open(pdb_path) as f:
        pdb_data = f.read()

    view = py3Dmol.view(width=800, height=500)
    view.addModel(pdb_data, "pdb")

    # Target: blue
    view.setStyle({"chain": "A"}, {"cartoon": {"color": "skyblue"}})
    # Binder: green
    view.setStyle({"chain": "B"}, {"cartoon": {"color": "lightgreen"}})

    view.zoomTo()
    view.show()

    print(f"\nBlue: target, Green: designed binder")